# 02 – Train pilot theo shard
**Cài đặt notebook:** Accelerator = **GPU T4 x2**; **Internet = On**; *Add Input* → output của notebook `01` (file `cremad_processed.tar`).

**Thứ tự chạy:**
1. **Đo thời gian (session tương tác):** đặt `TIMING_ONLY = True`, chạy từng cell. Cell đo thời gian in ra số giờ ước tính cho mỗi run và **số shard khuyến nghị**.
2. **Chạy thật:** đặt `TIMING_ONLY = False`, `N_SHARDS` = số shard khuyến nghị, `SHARD = 0`, rồi *Save Version → Save & Run All*. Lặp lại với `SHARD = 1, 2, …`; mỗi shard là một version.
3. Mỗi version cho ra `runs_shard{SHARD}.tar` (vài MB). **Tải file này ngay khi version chạy xong**, trước khi chạy shard tiếp theo, rồi gửi tất cả cho Claude.
   Nếu log báo `jobs not started because of the budget: N` với N > 0: upload `runs_shard{SHARD}.tar` thành một Kaggle Dataset, gắn Dataset đó làm input, rồi chạy lại **đúng shard đó**. Các run đã xong sẽ được bỏ qua.

In [ ]:
SHARD = 0            # 0 .. N_SHARDS-1
N_SHARDS = 4         # đặt theo kết quả đo thời gian
TIME_BUDGET_H = 8.0  # không bắt đầu job mới nếu nhiều khả năng vượt mốc này
TIMING_ONLY = True   # True = chỉ đo thời gian 3 epoch
GRID_ARGS = ''       # để trống = lưới pilot đầy đủ 49 run
EXTRA_TRAIN_ARGS = ''  # để trống trên Kaggle

In [ ]:
# Đường dẫn (giữ nguyên trên Kaggle)
WORK = '/tmp'               # đĩa tạm của session: code, data giải nén, run
KIN  = '/kaggle/input'      # input gắn vào notebook (chỉ đọc)
KOUT = '/kaggle/working'    # output được lưu cùng version

In [ ]:
import os, glob, subprocess, sys, json, math, importlib
sh = lambda cmd, check=True: subprocess.run(cmd, shell=True, check=check)
sh(f'rm -rf {WORK}/icme && git clone -q --depth 1 -b main https://github.com/ptrnghieu/icme.git {WORK}/icme')
os.chdir(f'{WORK}/icme/pilot')
print('code commit:', subprocess.run(f'git -C {WORK}/icme log -1 --format=%h_%cd', shell=True, capture_output=True, text=True).stdout)
print('CPUs:', os.cpu_count())
tars = glob.glob(f'{KIN}/**/cremad_processed.tar', recursive=True)
assert tars, 'Chưa gắn output của notebook 01 làm input'
os.makedirs(f'{WORK}/cremad', exist_ok=True)
sh(f'tar -xf {tars[0]} -C {WORK}/cremad')
os.makedirs(f'{WORK}/runs', exist_ok=True)
for t in glob.glob(f'{KIN}/**/runs_shard*.tar', recursive=True):   # khôi phục các run đã xong (nếu có)
    sh(f'tar -xf {t} -C {WORK}')
    print('restored', t)
WORKERS = max(1, os.cpu_count() // 2)
print('dataloader workers per job:', WORKERS)
sh('nvidia-smi --query-gpu=name,memory.total --format=csv', check=False)

In [ ]:
if TIMING_ONLY:
    # 2 job chạy song song (mỗi GPU một job), giống điều kiện khi chạy thật
    jobs = [f'--root {WORK}/cremad --out {WORK}/runs_timing --split standard --method ogm_ge --alpha 0.5 --seed {s} '
            f'--epochs 3 --amp --workers {WORKERS} {EXTRA_TRAIN_ARGS}' for s in (0, 1)]
    open(f'{WORK}/jobs_timing.txt', 'w').write('\n'.join(jobs) + '\n')
    sh(f'python launch.py --jobs {WORK}/jobs_timing.txt --gpus 0,1 --per_gpu 1')
    secs = []
    for f in glob.glob(f'{WORK}/runs_timing/*/log.jsonl'):
        recs = [json.loads(l) for l in open(f)]
        secs += [r['sec'] for r in recs[1:]]   # bỏ epoch đầu (khởi động)
    sec = sum(secs) / len(secs)
    h_run = 100 * sec / 3600 * 1.05            # 100 epoch, cộng 5% cho bước dump cuối
    shards = math.ceil(49 * h_run / (2 * TIME_BUDGET_H * 0.9))
    print(f'giây/epoch (2 job song song): {sec:.1f} | ước tính mỗi run: {h_run:.2f} h | '
          f'tổng 49 run trên 2 GPU: {49*h_run/2:.1f} h session | N_SHARDS khuyến nghị: {shards}')

In [ ]:
if not TIMING_ONLY:
    sh(f'python make_jobs.py --root {WORK}/cremad --out {WORK}/runs --jobs {WORK}/jobs.txt {GRID_ARGS} '
       f'--extra "--amp --workers {WORKERS} {EXTRA_TRAIN_ARGS}"')
    # --time_budget_h: không bắt đầu job mới nếu ước tính sẽ vượt mốc này
    # --hard_stop_h  : dừng hẳn các job còn chạy để notebook kịp lưu output trước giới hạn session của Kaggle
    # --snapshot_tar : ghi lại file tar sau mỗi job xong, để các run đã xong không bị mất
    sh(f'python launch.py --jobs {WORK}/jobs.txt --gpus 0,1 --per_gpu 1 --shard {SHARD} --n_shards {N_SHARDS} '
       f'--time_budget_h {TIME_BUDGET_H} --hard_stop_h {TIME_BUDGET_H + 0.5} --snapshot_tar {KOUT}/runs_shard{SHARD}.tar')
    sh(f"tar -cf {KOUT}/runs_shard{SHARD}.tar -C {WORK} --exclude='*.pt' runs")
    sh(f'python analyze.py --root {WORK}/cremad --runs {WORK}/runs --out {KOUT}/pilot_report_partial.md', check=False)
    sh(f'ls -la {KOUT}')